In [1]:
# Cell 1 — Install libraries
!pip install azure-storage-blob openpyxl pandas pyarrow -q

In [2]:
# Cell 2 — Konfigurasi & download file dari Blob Storage
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

# Ganti dengan storage account key kamu
STORAGE_ACCOUNT = "stfinalertai"
STORAGE_KEY = "YOUR_AZURE_STORAGE_KEY"
CONTAINER = "finalert-data"

# Connect ke Blob
blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=STORAGE_KEY
)
container_client = blob_service.get_container_client(CONTAINER)

def download_excel(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    data = blob_client.download_blob().readall()
    return io.BytesIO(data)

print("✅ Koneksi ke Blob Storage berhasil!")

✅ Koneksi ke Blob Storage berhasil!


In [3]:
# Cell 3 — Load Excel files dari Blob Storage
print("Loading file dari Blob Storage...")

# Load data_prep (sheet utama)
print("📥 Loading data_prep_datathon_dicoding6.xlsx...")
f1 = download_excel("raw/data prep datathon dicoding-6.xlsx")
xl_main = pd.ExcelFile(f1)
print(f"   Sheets: {xl_main.sheet_names}")

# Load TKD
print("📥 Loading TKD_per_Provinsi_2019_2025.xlsx...")
f2 = download_excel("raw/TKD_per_Provinsi_2019_2025.xlsx")
xl_tkd = pd.ExcelFile(f2)
print(f"   Sheets: {xl_tkd.sheet_names}")

print("\n✅ Kedua file berhasil diload!")

Loading file dari Blob Storage...
📥 Loading data_prep_datathon_dicoding6.xlsx...
   Sheets: ['Tingkat Penetrasi Internet', 'BI Rate', 'PDRB', 'PDRB Per Kapita', 'Tingkat Pengangguran Terbuka', 'Tab 9 (TWP90)', 'Tab 10 (Kualitas Pembiayaan)', 'Tab 5 (Dana Diberikan berdasark', 'Tab 7 (Penyaluran ke Sektor Pro', 'Inflasi per Provinsi Tahunan (Y', 'Realisasi TKD pertahun', 'Perbankan']
📥 Loading TKD_per_Provinsi_2019_2025.xlsx...
   Sheets: ['Total TKD per Provinsi', 'Komponen TKD per Provinsi', 'Panel TKD (JOIN-Ready)', 'Panduan Fitur TKD']

✅ Kedua file berhasil diload!


In [5]:
# Cek nama sheet yang benar
print(xl_main.sheet_names)

['Tingkat Penetrasi Internet', 'BI Rate', 'PDRB', 'PDRB Per Kapita', 'Tingkat Pengangguran Terbuka', 'Tab 9 (TWP90)', 'Tab 10 (Kualitas Pembiayaan)', 'Tab 5 (Dana Diberikan berdasark', 'Tab 7 (Penyaluran ke Sektor Pro', 'Inflasi per Provinsi Tahunan (Y', 'Realisasi TKD pertahun', 'Perbankan']


In [6]:
# Cell 4 (fixed) — Load semua sheet
df_twp90   = pd.read_excel(xl_main, sheet_name='Tab 9 (TWP90)')
df_bi      = pd.read_excel(xl_main, sheet_name='BI Rate')
df_pdrb    = pd.read_excel(xl_main, sheet_name='PDRB Per Kapita')
df_tpt     = pd.read_excel(xl_main, sheet_name='Tingkat Pengangguran Terbuka')
df_internet= pd.read_excel(xl_main, sheet_name='Tingkat Penetrasi Internet')
df_inflasi = pd.read_excel(xl_main, sheet_name='Inflasi per Provinsi Tahunan (Y')
df_bank    = pd.read_excel(xl_main, sheet_name='Perbankan')
df_tkd     = pd.read_excel(xl_tkd,  sheet_name='Panel TKD (JOIN-Ready)')

print("✅ Semua sheet berhasil dibaca!")
print(f"   TWP90    : {df_twp90.shape}")
print(f"   BI Rate  : {df_bi.shape}")
print(f"   PDRB     : {df_pdrb.shape}")
print(f"   TPT      : {df_tpt.shape}")
print(f"   Internet : {df_internet.shape}")
print(f"   Inflasi  : {df_inflasi.shape}")
print(f"   Perbankan: {df_bank.shape}")
print(f"   TKD      : {df_tkd.shape}")

✅ Semua sheet berhasil dibaca!
   TWP90    : (39, 181)
   BI Rate  : (12, 7)
   PDRB     : (38, 8)
   TPT      : (39, 11)
   Internet : (39, 7)
   Inflasi  : (113, 64)
   Perbankan: (193, 14)
   TKD      : (268, 12)


In [7]:
# Cell 5 — Preview struktur data & simpan ke cleaned/
print("Preview TWP90 (5 baris pertama):")
print(df_twp90.head())
print(f"\nKolom TWP90: {df_twp90.columns.tolist()}")

Preview TWP90 (5 baris pertama):
      Provinsi                                           Jan 2021  \
0          NaN  Jumlah Rekening Penerima Pinjaman Aktif (entitas)   
1       Banten                                             866729   
2  Dki Jakarta                                            9395728   
3   Jawa Barat                                            2978367   
4  Jawa Tengah                                             970575   

                         Unnamed: 2  Unnamed: 3  \
0  Outstanding Pinjaman (miliar Rp)  TWP 90 (%)   
1                           1405.41      0.0155   
2                           4590.21      0.0214   
3                           4021.88      0.0193   
4                           1183.11      0.0129   

                                       Februari 2021  \
0  Jumlah Rekening Penerima Pinjaman Aktif (entitas)   
1                                             838022   
2                                           10165291   
3                    

In [8]:
# Cell 6 — Reshape TWP90 ke long format
import numpy as np

# Row 0 adalah header kedua (nama variabel), row 1+ adalah data provinsi
# Skip row 0 (header), ambil data mulai row 1
df_raw = df_twp90.copy()

# Ambil nama provinsi dari kolom pertama (skip row 0)
provinsi_list = df_raw['Provinsi'].iloc[1:].values

# Setiap bulan = 3 kolom: n_rekening, outstanding, TWP90
# Total kolom = 1 (provinsi) + 60 bulan * 3 = 181 kolom
records = []
cols = df_raw.columns.tolist()

# Loop setiap bulan (mulai kolom index 1, step 3)
for i in range(1, len(cols), 3):
    periode_raw = cols[i]
    col_rekening = cols[i]
    col_outstanding = cols[i+1] if i+1 < len(cols) else None
    col_twp90 = cols[i+2] if i+2 < len(cols) else None
    
    for j, prov in enumerate(provinsi_list):
        row_idx = j + 1  # skip header row
        records.append({
            'provinsi': prov,
            'periode_raw': str(periode_raw),
            'n_rekening': df_raw.iloc[row_idx][col_rekening],
            'outstanding': df_raw.iloc[row_idx][col_outstanding] if col_outstanding else None,
            'twp90_pct': df_raw.iloc[row_idx][col_twp90] if col_twp90 else None,
        })

df_twp90_long = pd.DataFrame(records)
print(f"✅ TWP90 long format: {df_twp90_long.shape}")
print(df_twp90_long.head(10))

✅ TWP90 long format: (2280, 5)
                     provinsi periode_raw  n_rekening outstanding twp90_pct
0                      Banten    Jan 2021    866729.0     1405.41    0.0155
1                 Dki Jakarta    Jan 2021   9395728.0     4590.21    0.0214
2                  Jawa Barat    Jan 2021   2978367.0     4021.88    0.0193
3                 Jawa Tengah    Jan 2021    970575.0     1183.11    0.0129
4  Daerah Istimewa Yogyakarta    Jan 2021    139127.0      170.41    0.0334
5                  Jawa Timur    Jan 2021   1258463.0     1820.98    0.0189
6                        Aceh    Jan 2021     55316.0       69.49    0.0053
7              Sumatera Utara    Jan 2021    309142.0      364.71    0.0129
8              Sumatera Barat    Jan 2021    113695.0      148.54    0.0074
9                        Riau    Jan 2021    121246.0      152.96    0.0078


In [10]:
# Cell 7 — Bersihkan TWP90: fix periode & twp90_pct
import re
from datetime import datetime

bulan_map = {
    'jan':'01','januari':'01','februari':'02','maret':'03','april':'04',
    'mei':'05','juni':'06','juli':'07','agustus':'08','september':'09',
    'oktober':'10','november':'11','desember':'12'
}

def parse_periode(s):
    s = str(s).strip().lower()
    # Cek format datetime string (2022-04-01...)
    if s.startswith('20') and '-' in s:
        return s[:7] + '-01'
    # Format "Jan 2021", "Februari 2021", dll
    parts = s.split()
    if len(parts) == 2:
        bln = bulan_map.get(parts[0][:8], None)
        thn = parts[1][:4]
        if bln and thn.isdigit():
            return f"{thn}-{bln}-01"
    return None

df_twp90_long['periode'] = df_twp90_long['periode_raw'].apply(parse_periode)
df_twp90_long['twp90_pct'] = pd.to_numeric(
    df_twp90_long['twp90_pct'].astype(str).str.replace('%','').str.replace(',','.'),
    errors='coerce'
)
# Kalau nilai < 1, berarti desimal (0.0155) → kalikan 100
df_twp90_long['twp90_pct'] = df_twp90_long['twp90_pct'].apply(
    lambda x: x*100 if pd.notna(x) and x < 1 else x
)

df_clean = df_twp90_long.dropna(subset=['periode','twp90_pct']).copy()
print(f"✅ Setelah cleaning: {df_clean.shape}")
print(df_clean[['provinsi','periode','twp90_pct']].head(10))
print(f"\nRange TWP90: {df_clean['twp90_pct'].min():.2f}% - {df_clean['twp90_pct'].max():.2f}%")

✅ Setelah cleaning: (2078, 6)
                     provinsi     periode  twp90_pct
0                      Banten  2021-01-01       1.55
1                 Dki Jakarta  2021-01-01       2.14
2                  Jawa Barat  2021-01-01       1.93
3                 Jawa Tengah  2021-01-01       1.29
4  Daerah Istimewa Yogyakarta  2021-01-01       3.34
5                  Jawa Timur  2021-01-01       1.89
6                        Aceh  2021-01-01       0.53
7              Sumatera Utara  2021-01-01       1.29
8              Sumatera Barat  2021-01-01       0.74
9                        Riau  2021-01-01       0.78

Range TWP90: 0.03% - 11.58%


In [11]:
# Cell 8 — Simpan master_panel ke Blob Storage
import pyarrow as pa
import pyarrow.parquet as pq

# Simpan ke buffer dulu
buffer = io.BytesIO()
df_clean.to_parquet(buffer, index=False)
buffer.seek(0)

# Upload ke blob cleaned/
blob_client = container_client.get_blob_client("cleaned/twp90_clean.parquet")
blob_client.upload_blob(buffer, overwrite=True)

print(f"✅ twp90_clean.parquet berhasil diupload ke Blob Storage!")
print(f"   Path: finalert-data/cleaned/twp90_clean.parquet")
print(f"   Shape: {df_clean.shape}")

ArrowInvalid: ("Could not convert '3317.07' with type str: tried to convert to double", 'Conversion failed for column outstanding with type object')

In [12]:
# Cell 8 (fixed) — Bersihkan tipe data lalu simpan
df_clean['outstanding'] = pd.to_numeric(
    df_clean['outstanding'].astype(str).str.replace(',', '.').str.replace(' ', ''),
    errors='coerce'
)
df_clean['n_rekening'] = pd.to_numeric(df_clean['n_rekening'], errors='coerce')

# Simpan ke buffer
buffer = io.BytesIO()
df_clean.to_parquet(buffer, index=False)
buffer.seek(0)

# Upload ke blob cleaned/
blob_client = container_client.get_blob_client("cleaned/twp90_clean.parquet")
blob_client.upload_blob(buffer, overwrite=True)

print(f"✅ twp90_clean.parquet berhasil diupload!")
print(f"   Shape: {df_clean.shape}")
print(f"   Dtypes:\n{df_clean.dtypes}")

✅ twp90_clean.parquet berhasil diupload!
   Shape: (2078, 6)
   Dtypes:
provinsi        object
periode_raw     object
n_rekening     float64
outstanding    float64
twp90_pct      float64
periode         object
dtype: object


In [13]:
# Cell 9 — Proses BI Rate
print("Preview BI Rate:")
print(df_bi.head())
print(f"\nKolom: {df_bi.columns.tolist()}")

Preview BI Rate:
      Bulan  BI Rate 2020  BI Rate 2021  BI Rate 2022  BI Rate 2023  \
0   Januari          5.00          3.75           3.5          5.75   
1  Februari          4.75          3.50           3.5          5.75   
2     Maret          4.50          3.50           3.5          5.75   
3     April          4.50          3.50           3.5          5.75   
4       Mei          4.50          3.50           3.5          5.75   

   BI Rate 2024  BI Rate 2025  
0          6.00          5.75  
1          6.00          5.75  
2          6.00          5.75  
3          6.25          5.75  
4          6.25          5.75  

Kolom: ['Bulan', 'BI Rate 2020', 'BI Rate 2021', 'BI Rate 2022', 'BI Rate 2023', 'BI Rate 2024', 'BI Rate 2025']


In [14]:
# Cell 10 — Reshape BI Rate ke long format
bulan_map2 = {
    'januari':'01','februari':'02','maret':'03','april':'04',
    'mei':'05','juni':'06','juli':'07','agustus':'08',
    'september':'09','oktober':'10','november':'11','desember':'12'
}

records_bi = []
for _, row in df_bi.iterrows():
    bln = bulan_map2.get(row['Bulan'].lower().strip(), None)
    if not bln:
        continue
    for col in df_bi.columns[1:]:  # BI Rate 2020, 2021, dst
        thn = col.split()[-1]
        periode = f"{thn}-{bln}-01"
        records_bi.append({
            'periode': periode,
            'bi_rate': pd.to_numeric(row[col], errors='coerce')
        })

df_bi_long = pd.DataFrame(records_bi)
df_bi_long = df_bi_long.dropna(subset=['bi_rate'])
print(f"✅ BI Rate long format: {df_bi_long.shape}")
print(df_bi_long.head(10))

✅ BI Rate long format: (72, 2)
      periode  bi_rate
0  2020-01-01     5.00
1  2021-01-01     3.75
2  2022-01-01     3.50
3  2023-01-01     5.75
4  2024-01-01     6.00
5  2025-01-01     5.75
6  2020-02-01     4.75
7  2021-02-01     3.50
8  2022-02-01     3.50
9  2023-02-01     5.75


In [15]:
# Cell 11 — Preview PDRB Per Kapita
print(df_pdrb.head())
print(f"\nKolom: {df_pdrb.columns.tolist()}")
print(f"Shape: {df_pdrb.shape}")

         Provinsi  PDRB 2019    PDRB 2020    PDRB 2021  PDRB 2022  PDRB 2023  \
0            Aceh   30879000   31633000.0   34674000.0   38767000   41408000   
1  Sumatera Utara      54620   54979000.0   57442000.0   62922000   68306000   
2  Sumatera Barat   44886000   43826000.0   45218000.0   50264000   54334000   
3            Riau  111227000  114167000.0  129741000.0  151259000  154522000   
4           Jambi   60829000   57958000.0   64771000.0   76224000   79873000   

   PDRB 2024  PDRB 2025  
0   43782000   45770000  
1   73575000   78310000  
2   57083000   59549000  
3  165350000  176385000  
4   86775000   92786000  

Kolom: ['Provinsi', 'PDRB 2019', 'PDRB 2020', 'PDRB 2021', 'PDRB 2022', 'PDRB 2023', 'PDRB 2024', 'PDRB 2025']
Shape: (38, 8)


In [16]:
# Cell 12 — Reshape PDRB ke long format + interpolasi bulanan
records_pdrb = []
for _, row in df_pdrb.iterrows():
    prov = row['Provinsi']
    for col in df_pdrb.columns[1:]:
        thn = col.split()[-1]
        records_pdrb.append({
            'provinsi': prov,
            'periode': f"{thn}-01-01",
            'pdrb_per_kapita': pd.to_numeric(row[col], errors='coerce')
        })

df_pdrb_long = pd.DataFrame(records_pdrb).dropna(subset=['pdrb_per_kapita'])

# Interpolasi tahunan → bulanan per provinsi
pdrb_monthly = []
for prov in df_pdrb_long['provinsi'].unique():
    df_p = df_pdrb_long[df_pdrb_long['provinsi']==prov].copy()
    df_p['periode'] = pd.to_datetime(df_p['periode'])
    df_p = df_p.set_index('periode').resample('MS').interpolate('linear')
    df_p['provinsi'] = prov
    df_p = df_p.reset_index()
    pdrb_monthly.append(df_p)

df_pdrb_monthly = pd.concat(pdrb_monthly)
# Filter hanya 2021-2025
df_pdrb_monthly = df_pdrb_monthly[
    (df_pdrb_monthly['periode'] >= '2021-01-01') &
    (df_pdrb_monthly['periode'] <= '2025-12-01')
]
df_pdrb_monthly['periode'] = df_pdrb_monthly['periode'].dt.strftime('%Y-%m-%d')

print(f"✅ PDRB monthly: {df_pdrb_monthly.shape}")
print(df_pdrb_monthly.head(5))

✅ PDRB monthly: (1862, 3)
       periode provinsi  pdrb_per_kapita
24  2021-01-01     Aceh     3.467400e+07
25  2021-02-01     Aceh     3.501508e+07
26  2021-03-01     Aceh     3.535617e+07
27  2021-04-01     Aceh     3.569725e+07
28  2021-05-01     Aceh     3.603833e+07


In [17]:
# Cell 13 — Preview TPT, Internet, Inflasi
print("=== TPT ===")
print(df_tpt.head(3))
print(f"Kolom: {df_tpt.columns.tolist()}\n")

print("=== Internet ===")
print(df_internet.head(3))
print(f"Kolom: {df_internet.columns.tolist()}\n")

print("=== Inflasi ===")
print(df_inflasi.head(3))
print(f"Kolom: {df_inflasi.columns.tolist()}")

=== TPT ===
         Provinsi Februari 2021 Agustus 2021  Februari 2022  Agustus 2022  \
0            Aceh           6.3          6.3           5.97          6.17   
1  Sumatera Utara          6.01         6.33           5.47          6.16   
2  Sumatera Barat          6.67         6.52           6.17          6.28   

   Februari 2023 Agustus 2023  Februari 2024  Agustus 2024  Februari 2025  \
0           5.75         6.03           5.56          5.75           5.50   
1           5.24         5.89           5.10          5.60           5.05   
2           5.90         5.94           5.79          5.75           5.69   

   Agustus 2025  
0          5.64  
1          5.32  
2          5.62  
Kolom: ['Provinsi', 'Februari 2021', 'Agustus 2021', 'Februari 2022', 'Agustus 2022', 'Februari 2023', 'Agustus 2023', 'Februari 2024', 'Agustus 2024', 'Februari 2025', 'Agustus 2025']

=== Internet ===
   ID_Pemda      Unnamed: 1  Tingkat Penetrasi Internet  Unnamed: 3  \
0       NaN        Provi

In [18]:
# Cell 14 — Proses TPT (semesteran → interpolasi bulanan)
records_tpt = []
for _, row in df_tpt.iterrows():
    prov = row['Provinsi']
    for col in df_tpt.columns[1:]:
        parts = col.split()
        bln = bulan_map2.get(parts[0].lower(), None)
        thn = parts[1]
        if bln:
            records_tpt.append({
                'provinsi': prov,
                'periode': f"{thn}-{bln}-01",
                'tpt': pd.to_numeric(row[col], errors='coerce')
            })

df_tpt_long = pd.DataFrame(records_tpt).dropna(subset=['tpt'])

# Interpolasi semesteran → bulanan
tpt_monthly = []
for prov in df_tpt_long['provinsi'].unique():
    df_p = df_tpt_long[df_tpt_long['provinsi']==prov].copy()
    df_p['periode'] = pd.to_datetime(df_p['periode'])
    df_p = df_p.set_index('periode').resample('MS').interpolate('linear')
    df_p['provinsi'] = prov
    tpt_monthly.append(df_p.reset_index())

df_tpt_monthly = pd.concat(tpt_monthly)
df_tpt_monthly = df_tpt_monthly[
    (df_tpt_monthly['periode'] >= '2021-01-01') &
    (df_tpt_monthly['periode'] <= '2025-12-01')
]
df_tpt_monthly['periode'] = df_tpt_monthly['periode'].dt.strftime('%Y-%m-%d')
print(f"✅ TPT monthly: {df_tpt_monthly.shape}")

# Cell 14b — Proses Internet (tahunan → interpolasi bulanan)
# Fix header dulu
df_net = df_internet.copy()
df_net.columns = ['id_pemda','provinsi','2021','2022','2023','2024','2025']
df_net = df_net.iloc[1:].reset_index(drop=True)  # drop header row

records_net = []
for _, row in df_net.iterrows():
    prov = row['provinsi']
    for thn in ['2021','2022','2023','2024','2025']:
        records_net.append({
            'provinsi': prov,
            'periode': f"{thn}-01-01",
            'penetrasi_internet': pd.to_numeric(row[thn], errors='coerce')
        })

df_net_long = pd.DataFrame(records_net).dropna(subset=['penetrasi_internet'])

net_monthly = []
for prov in df_net_long['provinsi'].unique():
    df_p = df_net_long[df_net_long['provinsi']==prov].copy()
    df_p['periode'] = pd.to_datetime(df_p['periode'])
    df_p = df_p.set_index('periode').resample('MS').interpolate('linear')
    df_p['provinsi'] = prov
    net_monthly.append(df_p.reset_index())

df_net_monthly = pd.concat(net_monthly)
df_net_monthly = df_net_monthly[
    (df_net_monthly['periode'] >= '2021-01-01') &
    (df_net_monthly['periode'] <= '2025-12-01')
]
df_net_monthly['periode'] = df_net_monthly['periode'].dt.strftime('%Y-%m-%d')
print(f"✅ Internet monthly: {df_net_monthly.shape}")

# Cell 14c — Proses Inflasi (sudah bulanan)
bulan_abbr = {
    'jan':'01','feb':'02','mar':'03','apr':'04','mei':'05','jun':'06',
    'jul':'07','agu':'08','sept':'09','okt':'10','nov':'11','des':'12'
}

records_inf = []
for _, row in df_inflasi.iterrows():
    prov = row['Provinsi']
    for col in df_inflasi.columns[1:]:
        parts = col.lower().strip().split()
        if len(parts) == 2:
            bln = bulan_abbr.get(parts[0], None)
            thn = '20' + parts[1] if len(parts[1]) == 2 else parts[1]
            if bln:
                records_inf.append({
                    'provinsi': prov,
                    'periode': f"{thn}-{bln}-01",
                    'inflasi_mtm': pd.to_numeric(row[col], errors='coerce')
                })

df_inflasi_long = pd.DataFrame(records_inf).dropna(subset=['inflasi_mtm'])
df_inflasi_long = df_inflasi_long[
    (df_inflasi_long['periode'] >= '2021-01-01') &
    (df_inflasi_long['periode'] <= '2025-12-01')
]
print(f"✅ Inflasi long: {df_inflasi_long.shape}")

✅ TPT monthly: (2001, 3)
✅ Internet monthly: (1358, 3)
✅ Inflasi long: (2758, 3)


In [19]:
# Cell 15 — Preview Perbankan & TKD
print("=== Perbankan ===")
print(df_bank.head(3))
print(f"Kolom: {df_bank.columns.tolist()}\n")

print("=== TKD ===")
print(df_tkd.head(3))
print(f"Kolom: {df_tkd.columns.tolist()}")

=== Perbankan ===
  Identitas      Unnamed: 1 Unnamed: 2 Unnamed: 3          Kredit, DPK & NPL  \
0  Kode BPS        Provinsi      Tahun   Snapshot  Kredit Total\n(Miliar Rp)   
1        11            Aceh       2020   Des 2020                   36838.02   
2        12  Sumatera Utara       2020   Des 2020                  216540.16   

         Unnamed: 5 Unnamed: 6 Unnamed: 7        Unnamed: 8            UMKM  \
0  DPK\n(Miliar Rp)   LDR\n(%)   Zona LDR  NPL\n(Miliar Rp)  NPL Ratio\n(%)   
1           38359.8     0.9603      IDEAL            833.48          0.0226   
2          194939.1     1.1108     KRITIS           7183.92          0.0332   

                Unnamed: 10                 Unnamed: 11  \
0  Kredit UMKM\n(Miliar Rp)  Rasio UMKM\n(% dari total)   
1                      2945                      0.0799   
2                     54191                      0.2503   

                Unnamed: 12          Bank Kantor  
0  UMKM/KC\n(Miliar/kantor)  Jml KC\nBank (unit)  
1    

In [20]:
# Cell 16 — Proses Perbankan
# Fix header (row 0 adalah header asli)
df_bank.columns = ['kode_bps','provinsi','tahun','snapshot',
                   'kredit_total','dpk','ldr','zona_ldr',
                   'npl_miliar','npl_ratio','kredit_umkm',
                   'rasio_umkm','umkm_per_kc','jml_kc_bank']
df_bank = df_bank.iloc[1:].reset_index(drop=True)  # drop header row

df_bank['tahun'] = pd.to_numeric(df_bank['tahun'], errors='coerce')
df_bank['dpk'] = pd.to_numeric(df_bank['dpk'], errors='coerce')
df_bank['ldr'] = pd.to_numeric(df_bank['ldr'], errors='coerce')
df_bank['npl_ratio'] = pd.to_numeric(df_bank['npl_ratio'], errors='coerce')
df_bank['kredit_umkm'] = pd.to_numeric(df_bank['kredit_umkm'], errors='coerce')
df_bank['rasio_umkm'] = pd.to_numeric(df_bank['rasio_umkm'], errors='coerce')
df_bank['jml_kc_bank'] = pd.to_numeric(df_bank['jml_kc_bank'], errors='coerce')

# Interpolasi tahunan → bulanan per provinsi
bank_monthly = []
for prov in df_bank['provinsi'].unique():
    df_p = df_bank[df_bank['provinsi']==prov].copy()
    df_p['periode'] = pd.to_datetime(df_p['tahun'].astype(int).astype(str) + '-01-01')
    df_p = df_p.set_index('periode')[['dpk','ldr','npl_ratio','kredit_umkm','rasio_umkm','jml_kc_bank']]
    df_p = df_p.apply(pd.to_numeric, errors='coerce')
    df_p = df_p.resample('MS').interpolate('linear')
    df_p['provinsi'] = prov
    bank_monthly.append(df_p.reset_index())

df_bank_monthly = pd.concat(bank_monthly)
df_bank_monthly = df_bank_monthly[
    (df_bank_monthly['periode'] >= '2021-01-01') &
    (df_bank_monthly['periode'] <= '2025-12-01')
]
df_bank_monthly['periode'] = df_bank_monthly['periode'].dt.strftime('%Y-%m-%d')
print(f"✅ Perbankan monthly: {df_bank_monthly.shape}")

# Cell 16b — Proses TKD
# Fix header (row 1 adalah header asli)
df_tkd.columns = df_tkd.iloc[1].values
df_tkd = df_tkd.iloc[2:].reset_index(drop=True)
df_tkd.columns = ['kode_bps','provinsi','tahun','total_tkd',
                  'dbh_total','dau_total','dak_total','dana_spesial',
                  'dau_pct','dak_pct','dbh_pct','catatan']

for col in ['total_tkd','dbh_total','dau_total','dak_total','dana_spesial']:
    df_tkd[col] = pd.to_numeric(df_tkd[col], errors='coerce')
df_tkd['tahun'] = pd.to_numeric(df_tkd['tahun'], errors='coerce')
df_tkd['total_tkd_miliar'] = df_tkd['total_tkd'] / 1e9

# Interpolasi tahunan → bulanan
tkd_monthly = []
for prov in df_tkd['provinsi'].unique():
    df_p = df_tkd[df_tkd['provinsi']==prov].copy()
    df_p['periode'] = pd.to_datetime(df_p['tahun'].astype(int).astype(str) + '-01-01')
    df_p = df_p.set_index('periode')[['total_tkd_miliar','dbh_total','dau_total','dak_total']]
    df_p = df_p.resample('MS').interpolate('linear')
    df_p['provinsi'] = prov
    tkd_monthly.append(df_p.reset_index())

df_tkd_monthly = pd.concat(tkd_monthly)
df_tkd_monthly = df_tkd_monthly[
    (df_tkd_monthly['periode'] >= '2021-01-01') &
    (df_tkd_monthly['periode'] <= '2025-12-01')
]
df_tkd_monthly['periode'] = df_tkd_monthly['periode'].dt.strftime('%Y-%m-%d')
print(f"✅ TKD monthly: {df_tkd_monthly.shape}")

✅ Perbankan monthly: (1568, 8)
✅ TKD monthly: (1862, 6)


In [27]:
# Cell 17 — JOIN semua ke master panel
print("Menggabungkan semua data...")

# Base: TWP90 (38 provinsi x 60 bulan = 2280 baris)
df_master = df_clean[['provinsi','periode','n_rekening','outstanding','twp90_pct']].copy()

# Standardisasi nama provinsi (lowercase strip)
def clean_prov(s):
    return str(s).strip().lower()

df_master['_prov'] = df_master['provinsi'].apply(clean_prov)

# Join BI Rate (nasional, by periode saja)
df_bi_long['periode'] = df_bi_long['periode'].astype(str)
df_master = df_master.merge(df_bi_long, on='periode', how='left')

# Fungsi join per provinsi
def prep_join(df, prov_col='provinsi'):
    df = df.copy()
    df['_prov'] = df[prov_col].apply(clean_prov)
    df['periode'] = df['periode'].astype(str)
    return df

df_pdrb_j   = prep_join(df_pdrb_monthly)
df_tpt_j    = prep_join(df_tpt_monthly)
df_net_j    = prep_join(df_net_monthly)
df_inf_j    = prep_join(df_inflasi_long)
df_bank_j   = prep_join(df_bank_monthly)
df_tkd_j    = prep_join(df_tkd_monthly)

for df_join, cols in [
    (df_pdrb_j,  ['pdrb_per_kapita']),
    (df_tpt_j,   ['tpt']),
    (df_net_j,   ['penetrasi_internet']),
    (df_inf_j,   ['inflasi_mtm']),
    (df_bank_j,  ['dpk','ldr','npl_ratio','kredit_umkm','rasio_umkm','jml_kc_bank']),
    (df_tkd_j,   ['total_tkd_miliar','dbh_total','dau_total','dak_total']),
]:
    df_master = df_master.merge(
        df_join[['_prov','periode'] + cols],
        on=['_prov','periode'], how='left'
    )

df_master = df_master.drop(columns=['_prov'])

print(f"✅ Master panel: {df_master.shape}")
print(f"\nKolom: {df_master.columns.tolist()}")
print(f"\nMissing values:\n{df_master.isnull().sum()}")

Menggabungkan semua data...
✅ Master panel: (2280, 20)

Kolom: ['provinsi', 'periode', 'n_rekening', 'outstanding', 'twp90_pct', 'bi_rate', 'pdrb_per_kapita', 'tpt', 'penetrasi_internet', 'inflasi_mtm', 'dpk', 'ldr', 'npl_ratio', 'kredit_umkm', 'rasio_umkm', 'jml_kc_bank', 'total_tkd_miliar', 'dbh_total', 'dau_total', 'dak_total']

Missing values:
provinsi                0
periode                 0
n_rekening              0
outstanding             0
twp90_pct               0
bi_rate                 0
pdrb_per_kapita       418
tpt                   334
penetrasi_internet    922
inflasi_mtm             0
dpk                   712
ldr                   712
npl_ratio             712
kredit_umkm           712
rasio_umkm            712
jml_kc_bank           712
total_tkd_miliar      563
dbh_total             563
dau_total             563
dak_total             563
dtype: int64


In [28]:
# Cell 18 — Simpan master_panel ke Blob Storage
buffer2 = io.BytesIO()
df_master.to_parquet(buffer2, index=False)
buffer2.seek(0)

blob_client2 = container_client.get_blob_client("cleaned/master_panel.parquet")
blob_client2.upload_blob(buffer2, overwrite=True)

print(f"✅ master_panel.parquet berhasil diupload!")
print(f"   Shape  : {df_master.shape}")
print(f"   Kolom  : {len(df_master.columns)}")
print(f"   Periode: {df_master['periode'].min()} s/d {df_master['periode'].max()}")
print(f"   Provinsi: {df_master['provinsi'].nunique()} provinsi")
print(f"\nPreview:")
print(df_master[['provinsi','periode','twp90_pct','bi_rate','pdrb_per_kapita','tpt']].head(5))

✅ master_panel.parquet berhasil diupload!
   Shape  : (2280, 20)
   Kolom  : 20
   Periode: 2021-01-01 s/d 2025-12-01
   Provinsi: 38 provinsi

Preview:
  provinsi     periode  twp90_pct  bi_rate  pdrb_per_kapita  tpt
0     Aceh  2021-01-01       0.53     3.75     3.467400e+07  NaN
1     Aceh  2021-02-01       0.65     3.50     3.501508e+07  6.3
2     Aceh  2021-03-01       0.70     3.50     3.535617e+07  6.3
3     Aceh  2021-04-01       1.34     3.50     3.569725e+07  6.3
4     Aceh  2021-05-01       1.71     3.50     3.603833e+07  6.3


In [23]:
# FIX — Isi bulan yang hilang dengan interpolasi
import pandas as pd

print("Sebelum fix:")
print(f"  Total baris: {df_clean.shape[0]}")
print(f"  Periode unik: {df_clean['periode'].nunique()}")

# Buat semua kombinasi provinsi x periode yang lengkap
all_periods = pd.date_range('2021-01-01', '2025-12-01', freq='MS')
all_provs = df_clean['provinsi'].unique()

# Buat template lengkap
template = pd.MultiIndex.from_product(
    [all_provs, all_periods], 
    names=['provinsi', 'periode']
).to_frame(index=False)

# Merge dengan data yang ada
df_clean['periode'] = pd.to_datetime(df_clean['periode'])
df_full = template.merge(df_clean, on=['provinsi','periode'], how='left')

# Interpolasi per provinsi untuk mengisi gap
df_full = df_full.sort_values(['provinsi','periode'])
for col in ['twp90_pct', 'n_rekening', 'outstanding']:
    df_full[col] = df_full.groupby('provinsi')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

df_full['periode'] = df_full['periode'].dt.strftime('%Y-%m-%d')

print(f"\nSetelah fix:")
print(f"  Total baris: {df_full.shape[0]}")
print(f"  Periode unik: {df_full['periode'].nunique()}")
print(f"  Expected: 38 x 60 = {38*60}")

# Cek September 2021 sudah ada
sept_check = df_full[df_full['periode']=='2021-09-01'][['provinsi','twp90_pct']].head(3)
print(f"\nSeptember 2021 sample:")
print(sept_check)

Sebelum fix:
  Total baris: 2078
  Periode unik: 59


OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 202-03-01 00:00:00 present at position 49

In [24]:
# FIX — Bersihkan periode invalid dulu lalu isi bulan yang hilang
import pandas as pd

# Fix periode invalid dulu
df_clean['periode'] = pd.to_datetime(df_clean['periode'], errors='coerce')
df_clean = df_clean.dropna(subset=['periode'])
df_clean = df_clean[df_clean['periode'].dt.year >= 2021]

print(f"Setelah clean: {df_clean.shape}")
print(f"Periode unik: {df_clean['periode'].nunique()}")

# Buat template lengkap 38 provinsi x 60 bulan
all_periods = pd.date_range('2021-01-01', '2025-12-01', freq='MS')
all_provs = df_clean['provinsi'].unique()

template = pd.MultiIndex.from_product(
    [all_provs, all_periods],
    names=['provinsi', 'periode']
).to_frame(index=False)

# Merge
df_full = template.merge(df_clean, on=['provinsi','periode'], how='left')
df_full = df_full.sort_values(['provinsi','periode'])

# Interpolasi untuk isi gap
for col in ['twp90_pct', 'n_rekening', 'outstanding']:
    df_full[col] = df_full.groupby('provinsi')[col].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )

print(f"\nSetelah fix:")
print(f"  Total baris : {df_full.shape[0]}")
print(f"  Periode unik: {df_full['periode'].nunique()}")
print(f"  Expected    : 38 x 60 = {38*60}")

# Cek bulan yang hilang sudah terisi
sept = df_full[df_full['periode']=='2021-09-01'][['provinsi','twp90_pct']].head(3)
mar = df_full[df_full['periode']=='2025-03-01'][['provinsi','twp90_pct']].head(3)
print(f"\nSeptember 2021:")
print(sept)
print(f"\nMaret 2025:")
print(mar)

Setelah clean: (2040, 6)
Periode unik: 58

Setelah fix:
  Total baris : 2280
  Periode unik: 60
  Expected    : 38 x 60 = 2280

September 2021:
     provinsi  twp90_pct
368      Aceh      2.395
1628     Bali      1.940
8      Banten      1.770

Maret 2025:
     provinsi  twp90_pct
410      Aceh      1.155
1670     Bali      1.385
50     Banten      2.505


In [25]:
# Update df_clean dengan yang sudah lengkap
df_full['periode'] = df_full['periode'].dt.strftime('%Y-%m-%d')
df_clean = df_full.copy()

# Simpan ulang twp90_clean.parquet
buffer = io.BytesIO()
df_clean.to_parquet(buffer, index=False)
buffer.seek(0)
blob_client = container_client.get_blob_client("cleaned/twp90_clean.parquet")
blob_client.upload_blob(buffer, overwrite=True)
print(f"✅ twp90_clean.parquet updated: {df_clean.shape}")

# Cek distribusi
print(f"\nStats TWP90 setelah fix:")
print(f"  Min  : {df_clean['twp90_pct'].min():.2f}%")
print(f"  Max  : {df_clean['twp90_pct'].max():.2f}%")
print(f"  Mean : {df_clean['twp90_pct'].mean():.2f}%")
print(f"  NaN  : {df_clean['twp90_pct'].isna().sum()}")

✅ twp90_clean.parquet updated: (2280, 6)

Stats TWP90 setelah fix:
  Min  : 0.03%
  Max  : 11.58%
  Mean : 1.77%
  NaN  : 0


In [26]:
# Update master panel dengan TWP90 yang sudah fix
df_master = df_clean[['provinsi','periode','n_rekening','outstanding','twp90_pct']].copy()

def clean_prov(s):
    return str(s).strip().lower()

df_master['_prov'] = df_master['provinsi'].apply(clean_prov)
df_master['periode'] = df_master['periode'].astype(str)

# Join BI Rate
df_bi_long['periode'] = df_bi_long['periode'].astype(str)
df_master = df_master.merge(df_bi_long, on='periode', how='left')

# Join semua variabel lain
def prep_join(df, prov_col='provinsi'):
    df = df.copy()
    df['_prov'] = df[prov_col].apply(clean_prov)
    df['periode'] = df['periode'].astype(str)
    return df

df_pdrb_j  = prep_join(df_pdrb_monthly)
df_tpt_j   = prep_join(df_tpt_monthly)
df_net_j   = prep_join(df_net_monthly)
df_inf_j   = prep_join(df_inflasi_long)
df_bank_j  = prep_join(df_bank_monthly)
df_tkd_j   = prep_join(df_tkd_monthly)

for df_join, cols in [
    (df_pdrb_j, ['pdrb_per_kapita']),
    (df_tpt_j,  ['tpt']),
    (df_net_j,  ['penetrasi_internet']),
    (df_inf_j,  ['inflasi_mtm']),
    (df_bank_j, ['dpk','ldr','npl_ratio','kredit_umkm','rasio_umkm','jml_kc_bank']),
    (df_tkd_j,  ['total_tkd_miliar','dbh_total','dau_total','dak_total']),
]:
    df_master = df_master.merge(
        df_join[['_prov','periode'] + cols],
        on=['_prov','periode'], how='left'
    )

df_master = df_master.drop(columns=['_prov'])

print(f"✅ Master panel updated: {df_master.shape}")
print(f"\nMissing values:")
print(df_master.isnull().sum())

# Simpan
buffer = io.BytesIO()
df_master.to_parquet(buffer, index=False)
buffer.seek(0)
container_client.get_blob_client("cleaned/master_panel.parquet").upload_blob(buffer, overwrite=True)
print("\n✅ master_panel.parquet updated!")

✅ Master panel updated: (2280, 20)

Missing values:
provinsi                0
periode                 0
n_rekening              0
outstanding             0
twp90_pct               0
bi_rate                 0
pdrb_per_kapita       418
tpt                   334
penetrasi_internet    922
inflasi_mtm             0
dpk                   712
ldr                   712
npl_ratio             712
kredit_umkm           712
rasio_umkm            712
jml_kc_bank           712
total_tkd_miliar      563
dbh_total             563
dau_total             563
dak_total             563
dtype: int64

✅ master_panel.parquet updated!
